# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w04_signal_audit.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** One row = one content page; the score ranks
which page an editor should review first. This notebook audited the beliefs behind FlyRank's
flags *before* trusting them: distributions first, then one mini-test per signal, each ending in
an honest verdict (CONFIRMED / OPPOSITE / MIXED / FALSE), then the flag-linked test.

Skill: `auditing-signals` + `flyrank/flyrank-data` (loaded from `skills/README.md`).

> The label `is_declining_label = (trend_direction == "down")` is used **only for evaluation** of
> a test. It is never a feature and no flag relies on it in this notebook.


## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root
and loads the starter slice.


In [1]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542


## 1. Distributions

*Look before deciding.* Traffic fields here are heavy-tailed — a few giants and a long tail of
tiny pages. That fact changes everything below: I bucket and rank instead of trusting a raw
Pearson number, and I show row counts next to every rate.


In [2]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# --- distributions of the fields we are about to test ---
KEY = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "word_count",
       "content_age_days", "days_since_last_update", "engagement_rate",
       "scroll_rate", "ai_traffic_pct"]
print("Key fields (count, mean, median, p90, min, max):")
print(df[KEY].describe(percentiles=[.5, .9]).T.round(2).to_string())
print()

imp = df["impressions_90d"]
print("Heavy tail check (impressions_90d):")
print(f"  mean/median ratio: {imp.mean()/imp.median():,.1f}")
print(f"  rows below 500 impressions: {((imp < 500).mean() * 100):.1f}%")
print(f"  rows at 30k+ impressions: {((imp >= 30000).mean() * 100):.2f}%")
print(f"  top 1% of pages hold {imp.nlargest(int(len(imp) * .01)).sum() / imp.sum() * 100:.1f}% of all impressions")
print(f"  skew: raw {imp.skew():.1f}  vs  log1p {np.log1p(imp).skew():.1f}")
print()

print("Gotchas that change how I test:")
print(f"  'ratio > 100' is real, not a bug: scroll_rate > 100 on {int((df['scroll_rate'] > 100).sum()):,} rows, "
      f"ai_traffic_pct > 100 on {int((df['ai_traffic_pct'] > 100).sum()):,} (denominators from different systems)")
print(f"  avg_position == 0 means 'no data', not rank 0: {int((df['avg_position'] == 0).sum()):,} rows")
print(f"  word_count missing on {df['word_count'].isna().mean()*100:.1f}% of rows — missingness follows content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: f"{s.isna().mean()*100:.1f}% missing").to_string())
print()

print("Why a raw correlation is a trap here:")
raw = df[["impressions_90d", "clicks_90d"]].corr().iloc[0, 1]
log_ = np.log1p(df[["impressions_90d", "clicks_90d"]]).corr().iloc[0, 1]
spearman = df[["impressions_90d", "clicks_90d"]].corr(method="spearman").iloc[0, 1]
print(f"  corr(impressions, clicks) — Pearson raw: {raw:.3f} | Pearson on log1p: {log_:.3f} | Spearman: {spearman:.3f}")
print("  => I never trust a Pearson number on raw traffic. Every test below uses buckets/tiers with n shown.")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
Key fields (count, mean, median, p90, min, max):
                          count     mean       std   min      50%       90%       max
impressions_90d         30000.0  5200.37  16838.02   1.0   731.00  12136.40  517715.0
clicks_90d              30000.0    16.10     75.08   0.0     1.00     32.00    4178.0
ctr                     30000.0     0.51      3.28   0.0     0.07      0.65     100.0
avg_position            30000.0    16.34     15.22   0.0    10.80     36.80     245.0
word_count              22301.0  3107.76   1452.38   8.0  2877.00   5327.00    9546.0
content_age_days        30000.0   256.17    132.71  90.0   236.00    463.00     564.0
days_since_last_update  30000.0    46.10     42.08   1.0    20.00    104.00     373.0
engagement_rate         30000.0     2.53      8.31   0.0     0.00      6.94     100.0
scroll_rate             29875.0    18.21     29.47   0.0     5.00     50.00     300.0
ai_traffic_pct    

**Reading the distributions:** `impressions_90d` is strongly right-skewed (mean ~7x the
median; the top 1% of pages hold ~25% of all impressions; skew drops from 11 to ~0 after
`log1p`). Clicks are tiny (median 1). `ctr` is a x100 percentage with a long right tail up to 100.
Two measurement gotchas are real: `scroll_rate` and `ai_traffic_pct` exceed 100 on some rows
(denominators come from different systems — read as-is, not as a bug), and `avg_position = 0`
means "no position data" (1,205 rows), so position tests keep a `has_position` filter. Missingness
in `word_count` follows `content_type` (keyword articles lose ~28%), so the audit never blind-fills —
and neither should the model.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three safe signals, each with a mini-test and a verdict. All are observable at export time —
none touches the trend windows that make the label.


### Signal test #1 — CTR underperforming at a good position (behind the CTR-fix flag)

**Claim:** *among pages that reach page 1-2, the ones that click far below the field are more
likely to be declining.* If this is false, the CTR-fix flag is flagging noise.


In [3]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# Signal test #1 — CTR vs position (behind the CTR-fix flag)
# Claim: "Among pages that reach page 1-2, the ones that click far below the field
#         are more likely to be declining."
pos12 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
pos12["band"] = pd.cut(pos12["ctr"], [0, 0.1, 0.5, 1.5, 100], labels=["<0.1", "0.1-0.5", "0.5-1.5", ">1.5"])
t = pos12.groupby("band", observed=True)["is_declining_label"].agg(n="size", rate="mean").round(3)
print(f"Slice: position 1-20 (n = {len(pos12):,}; overall declining rate {pos12['is_declining_label'].mean():.3f})")
print("  declining rate as CTR falls below the field:")
print(t.to_string())
print()

# denominator honesty: band CTR from totals, not the mean of per-page rates
w = pos12.groupby("band", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "declining_rate": g["is_declining_label"].mean(),
        "ctr_from_totals": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100,
    }), include_groups=False).round(3)
print("Same bands, CTR recomputed from row totals (clicks/impressions x100) — the true field rate:")
print(w.to_string())
print()

# rerun on a different slice: mid-volume pages, position 1-20
sub = df[(df["impressions_90d"] >= 500) & (df["impressions_90d"] < 3000) &
         (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
sub["band"] = pd.cut(sub["ctr"], [0, 0.1, 0.5, 1.5, 100], labels=["<0.1", "0.1-0.5", "0.5-1.5", ">1.5"])
s2 = sub.groupby("band", observed=True)["is_declining_label"].agg(n="size", rate="mean").round(3)
print(f"Cross-slice check — impressions 500-2,999, position 1-20 (n = {len(sub):,}):")
print(s2.to_string())
print()
print("VERDICT: CONFIRMED — declining rate rises monotonically as CTR falls (0.42 -> 0.67 on the")
print("main slice; 0.47 -> 0.70 on the cross-slice). Every cell has n > 50. The CTR-fix flag's")
print("core assumption is real.")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
Slice: position 1-20 (n = 20,256; overall declining rate 0.580)
  declining rate as CTR falls below the field:
            n   rate
band                
<0.1     1916  0.672
0.1-0.5  7233  0.594
0.5-1.5  2699  0.509
>1.5      920  0.418

Same bands, CTR recomputed from row totals (clicks/impressions x100) — the true field rate:
              n  declining_rate  ctr_from_totals
band                                            
<0.1     1916.0           0.672            0.064
0.1-0.5  7233.0           0.594            0.262
0.5-1.5  2699.0           0.509            0.775
>1.5      920.0           0.418            2.072

Cross-slice check — impressions 500-2,999, position 1-20 (n = 5,717):
            n   rate
band                
<0.1      820  0.702
0.1-0.5  2944  0.647
0.5-1.5   754  0.560
>1.5       68  0.471

VERDICT: CONFIRMED — declining rate rises monotonically as CTR falls (0.42 -> 0.67 on the
main slice; 0.4

### Signal test #2 — staleness / freshness (behind the refresh flags)

**Claim:** *pages not updated in a long time are more likely to be declining, and freshly-updated
pages decline less.* The refresh flags assume old = at risk.


In [4]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# Signal test #2 — staleness / freshness (behind the refresh flags)
# Claim: "Pages not updated in a long time are more likely to be declining;
#         freshly-updated pages decline less."
vis = df[df["impressions_90d"] >= 300].copy()
vis["stale_band"] = pd.cut(vis["days_since_last_update"], [0, 30, 90, 103, 179, 10**6],
                           labels=["0-30", "31-90", "91-103", "104-179", "180+"])
t = vis.groupby("stale_band", observed=True)["is_declining_label"].agg(n="size", rate="mean").round(3)
print(f"Slice: visible pages (impressions >= 300; n = {len(vis):,}; overall rate {vis['is_declining_label'].mean():.3f})")
print(t.to_string())
print()
print("  *** sample-size floors: '91-103' (n=40) and '180+' (n=22) sit BELOW the ~50-row floor —")
print("      those cells get no verdict, they are reports only. (Saying 'insufficient data' is a finding.)")
print()

# rerun on a different slice: page 1-2 pages only
st2 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
st2["stale_band"] = pd.cut(st2["days_since_last_update"], [0, 30, 90, 103, 179, 10**6],
                           labels=["0-30", "31-90", "91-103", "104-179", "180+"])
t2 = st2.groupby("stale_band", observed=True)["is_declining_label"].agg(n="size", rate="mean").round(3)
print(f"Cross-slice check — position 1-20 (n = {len(st2):,}; overall rate {st2['is_declining_label'].mean():.3f}):")
print(t2.to_string())
print()
print("VERDICT: MIXED — the relationship is not monotonic. Fresh pages do NOT decline less")
print("(0-30 sits at the overall rate on both slices), and only the 104-179 tail is consistently")
print("elevated (0.62 / 0.64). Rule design: gate staleness at the far tail only — never a general")
print("'old page = at risk' signal.")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
Slice: visible pages (impressions >= 300; n = 18,752; overall rate 0.595)
                n   rate
stale_band              
0-30        11398  0.581
31-90         120  0.558
91-103         40  0.250
104-179      7172  0.619
180+           22  0.818

  *** sample-size floors: '91-103' (n=40) and '180+' (n=22) sit BELOW the ~50-row floor —
      those cells get no verdict, they are reports only. (Saying 'insufficient data' is a finding.)

Cross-slice check — position 1-20 (n = 20,256; overall rate 0.580):
                n   rate
stale_band              
0-30        14028  0.559
31-90         122  0.557
91-103        178  0.506
104-179      5796  0.635
180+          132  0.515

VERDICT: MIXED — the relationship is not monotonic. Fresh pages do NOT decline less
(0-30 sits at the overall rate on both slices), and only the 104-179 tail is consistently
elevated (0.62 / 0.64). Rule design: gate staleness at the far tail 

### Signal test #3 — visibility: are buried pages the ones declining?

**Claim:** *pages buried in deep search results are the ones in trouble.* This is the quiet
counter-check to the "invisible == broken" assumption.


In [5]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# Signal test #3 — visibility: are buried pages the ones in decline?
# Claim: "Pages buried in deep search results are the ones in trouble."
haspos = df[df["avg_position"] > 0].copy()  # avg_position == 0 is 'no data', not rank 0
t = haspos.groupby("position_tier", observed=True)["is_declining_label"].agg(n="size", rate="mean").round(3)
print(f"Slice: pages with a measured position (n = {len(haspos):,})")
print("  position_tier: top_3 <= 3 | page_1 <= 10 | striking <= 20 | page_3_5 <= 50 | deep > 50")
print(t.to_string())
print()
print("VERDICT: OPPOSITE — the pages nobody sees (deep, position > 50) are the LEAST declining")
print("(0.34), not the most. Declining pressure concentrates in the visible middle band")
print("(striking 0.61, n = 7,304; page_1 0.57). 'Less visible = broken' is not supported by this")
print("slice; visibility alone should never be a risk flag.")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
Slice: pages with a measured position (n = 28,795)
  position_tier: top_3 <= 3 | page_1 <= 10 | striking <= 20 | page_3_5 <= 50 | deep > 50
                   n   rate
position_tier              
deep            1319  0.344
page_1         11814  0.570
page_3_5        7242  0.562
striking        7304  0.610
top_3           1116  0.494

VERDICT: OPPOSITE — the pages nobody sees (deep, position > 50) are the LEAST declining
(0.34), not the most. Declining pressure concentrates in the visible middle band
(striking 0.61, n = 7,304; page_1 0.57). 'Less visible = broken' is not supported by this
slice; visibility alone should never be a risk flag.


## 3. The flag-linked test

**Which flag?** the **CTR-fix flag** — my baseline rule in `w04_baseline_score` fires
`refresh_and_review_ctr` on exactly this population (visible AND position 1-20 AND low CTR), so
this audit checks that rule's premise on the flag's own rows. The refresh (staleness) flag is the
second premise checked.


In [6]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# Section 3 — the flag-linked test: does the CTR-fix flag's assumption hold?
# FlyRank's CTR-fix flag fires on: visible (impressions >= 300) AND position 1-20 AND ctr < 0.5.
# My baseline rule in w04 uses this exact population, so this audit checks its premise directly.
on12 = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
flag = (df["impressions_90d"] >= 300) & on12 & (df["ctr"] < 0.5)

n_flag = int(flag.sum())
rate_flag = df.loc[flag, "is_declining_label"].mean()
not_dec = int((~df.loc[flag, "is_declining_label"].astype(bool)).sum())

zero = (df["impressions_90d"] >= 300) & on12 & (df["ctr"] == 0)
print(f"CTR-fix flag population: n = {n_flag:,}")
print(f"  declining rate: {rate_flag:.3f}   (base rate {df['is_declining_label'].mean():.3f}; "
      f"pos-1-20 slice rate {df.loc[on12, 'is_declining_label'].mean():.3f})")
print(f"  NOT declining inside the flag (over-flag risk): n = {not_dec:,} ({not_dec/n_flag*100:.0f}%)")
print(f"  zero-click subset (ctr == 0): n = {int(zero.sum()):,}, rate = {df.loc[zero, 'is_declining_label'].mean():.3f}")
print()

# The refresh flag from the baseline rule: visible AND position 1-20 AND >= 104 days untouched.
refresh = (df["impressions_90d"] >= 300) & on12 & (df["days_since_last_update"] >= 104)
print(f"Refresh flag population (visible, pos 1-20, >= 104 days): n = {int(refresh.sum()):,}, "
      f"rate = {df.loc[refresh, 'is_declining_label'].mean():.3f}  "
      f"(base {df['is_declining_label'].mean():.3f})")
print()
print("FLAG-LINKED VERDICTS:")
print("  CTR-fix flag : CONFIRMED — its assumption holds: 0.64 vs a 0.54 base, and the zero-click")
print("                 extreme runs 0.76. But ~1/3 of flagged pages are NOT declining, so the flag")
print("                 is a screen, not a verdict — which is exactly why the baseline adds the")
print("                 human top-10 review step.")
print("  Refresh flag : MIXED — the 104+ day population is only mildly elevated (0.63 vs 0.54).")
print("                 It is kept as a gated bonus, never a primary flag.")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
CTR-fix flag population: n = 10,730
  declining rate: 0.635   (base rate 0.542; pos-1-20 slice rate 0.580)
  NOT declining inside the flag (over-flag risk): n = 3,912 (36%)
  zero-click subset (ctr == 0): n = 1,826, rate = 0.762

Refresh flag population (visible, pos 1-20, >= 104 days): n = 4,719, rate = 0.629  (base 0.542)

FLAG-LINKED VERDICTS:
  CTR-fix flag : CONFIRMED — its assumption holds: 0.64 vs a 0.54 base, and the zero-click
                 extreme runs 0.76. But ~1/3 of flagged pages are NOT declining, so the flag
                 is a screen, not a verdict — which is exactly why the baseline adds the
                 human top-10 review step.
  Refresh flag : MIXED — the 104+ day population is only mildly elevated (0.63 vs 0.54).
                 It is kept as a gated bonus, never a primary flag.


## 4. What this means in practice

**Two or three sentences a content team can act on.** Measured on this 30k-page slice (32
pseudonymized clients, trailing-90-day window), the strongest audited signal is *CTR at a good
position*: a page on page 1-2 that under-clicks is more likely to be declining (0.42 → 0.67 as
CTR falls; zero-click pages run 0.76), so that cohort is the most defensible first-daily-review
queue. Staleness is weaker than it sounds — fresh and moderately-stale pages decline at the same
rate, and only the 104+ day tail earns a bump — so a refresh queue should lean on the CTR gap and
use "old page" as a gated bonus, never a primary trigger. The surprise is signal #3: deep,
barely-visible pages are the *least* declining, so editorial review capacity belongs on visible
pages that are losing clicks, not on invisible ones.

These are observed associations (decision-support for ranking), not causal claims: refreshing
still requires a human look, and a future-window design (w05+) is what would let us say anything
about what happens *after* an edit.


In [7]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

# Section 4 — the audit in one table
summary = pd.DataFrame([
    {"signal": "CTR at a good position (pos 1-20)", "slice/n": "pos 1-20, n=20,256 (min cell 920)",
     "declining rate range": "0.42 -> 0.67", "verdict": "CONFIRMED"},
    {"signal": "Staleness / freshness", "slice/n": "visible + pos 1-20, n=18,752 (2 cells < 50-row floor)",
     "declining rate range": "0.51-0.64 (non-monotonic; only 104-179d tail elevated)", "verdict": "MIXED"},
    {"signal": "Visibility (position tier)", "slice/n": "has-position, n=28,795",
     "declining rate range": "deep 0.34 < visible 0.61", "verdict": "OPPOSITE to the claim"},
])
print(summary.to_string(index=False))
print()
print("Flag-linked: CTR-fix CONFIRMED (0.64 vs 0.54 base; zero-click subset 0.76)")
print("             refresh MIXED (0.63 vs 0.54 base; gated at 104+ days).")


Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542
                           signal                                               slice/n                                   declining rate range               verdict
CTR at a good position (pos 1-20)                     pos 1-20, n=20,256 (min cell 920)                                           0.42 -> 0.67             CONFIRMED
            Staleness / freshness visible + pos 1-20, n=18,752 (2 cells < 50-row floor) 0.51-0.64 (non-monotonic; only 104-179d tail elevated)                 MIXED
       Visibility (position tier)                                has-position, n=28,795                               deep 0.34 < visible 0.61 OPPOSITE to the claim

Flag-linked: CTR-fix CONFIRMED (0.64 vs 0.54 base; zero-click subset 0.76)
             refresh MIXED (0.63 vs 0.54 base; gated at 104+ days).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Every verdict has a table with visible n's under it (and sample-size floors respected)
- [x] Triangle: heavy tails handled (buckets/tiers, no raw-Pearson verdicts)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
